# CSC 349A Numerical Analysis - Lecture 23 - Romberg Integration

Romberg integration is the application of Richardson's extrapolation to composite trapezoid rule approximations. Let $I_{k,1}$ denote the composite trapezoid rule approximation to $\int_a^b f(x)dx$ with $2^{k-1}$ subintervals, using $h=\frac{b-a}{2^{k-l}}$. Then, letting $f_i = f(x_i)$,

\begin{align*}
I_{1,1} &= \frac{h}{2}\left[f_0+f_1\right], \text{ where } h = b-a \\
I_{2,1} &= \frac{h}{2}\left[f_0+2f_1+f_2\right], \text{ where } h = \frac{b-a}{2} \\
I_{3,1} &= \frac{h}{2}\left[f_0+2f_1+2f_2+2f_3+f_4\right], \text{ where } h = \frac{b-a}{4}
\end{align*}
etc.

Turns out that the error term for the composite trapezoid rule, $-\frac{b-a}{12}h^2f''(\mu)$ with $a<\mu<b$, has a series expansion of the form,

$$
K_1h^2+K_2h^4+K_3h^6+\dotsm,
$$

where each $K_i$ is a constant independent of $h$, thus R.E. applies. For example, letting $h=b-a$,

\begin{align*}
\int_a^b f(x)dx &= I_{1,1} +  K_1h^2+K_2h^4+K_3h^6+\dotsm \\
\int_a^b f(x)dx &= I_{2,1} +  K_1\frac{h^2}{4}+K_2\frac{h^4}{16}+K_3\frac{h^6}{64}+\dotsm
\end{align*}
etc.

## Romberg Integration Table
Thus, we have the comparable Romberg table

| $O(h^2)$ | $O(h^4)$ | $O(h^6)$ | $O(h^8)$ |
|:---------|:---------|:---------|:---------|
| $I_{1,1}$ | | | | 
| $I_{2,1}$ | $I_{2,2}$ | | | 
| $I_{3,1}$ | $I_{3,2}$ | $I_{3,3}$ | | 
| $I_{4,1}$ | $I_{4,2}$ | $I_{4,3}$ | $I_{4,4}$ |

In general, entries are computed by,

$$
I_{k,j} = I_{k,j-1} + \frac{I_{k,j-1}-I_{k-1,j-1}}{4^{j-1}-1},
$$

for $k=2,3,4,\dots$ and $j=2,3,\dots,k$.

### Example of Romberg Integration
Approximate $\int_1^3 \frac{1}{x}dx$ using Romberg integration. So, to start we calculate $I_{1,1}$ with $h=b-a=3-1=2$, (i.e single application of the trap rule)

$$
I_{1,1} = \frac{h}{2}[f(1)+f(3)]=\frac{2}{2}\left[\frac{1}{1}+\frac{1}{3}\right]=\frac{4}{3}=1.333333
$$

Now, for $I_{2,1}$, we are doing two applications of the trap rule with $h=\frac{3-1}{2}=1$,

$$
I_{2,1} = \frac{h}{2}[f(1)+2f(2)+f(3)]=\frac{1}{2}\left[\frac{1}{1}+2\left(\frac{1}{2}\right)+\frac{1}{3}\right]=\frac{7}{6}=1.166667
$$

Then we apply R.E. to get $I_{2,2}$,

$$
I_{2,2}=I_{2,1}+\frac{I_{2,1}-I_{1,1}}{3}=1.166667+\frac{1.166667-1.333333}{3}=1.111111
$$

etc.

The Romberg table is as follows:

| $h$ | $I_{k,1}$ | $I_{k,2}$ | $I_{k,3}$ | $I_{k,4}$ | $I_{k,5}$ |
|:---------|:---------|:---------|:---------|:---------|:---------|
| $h=2$ | $1.333333$ | | | | |
| $h=1$ | $1.166667$ | $1.111111$ | | | |
| $h=0.5$ | $1.116667$ | $1.100000$ | $1.099259$ | | |
| $h=0.25$ | $1.103211$ | $1.098726$ | $1.098641$ | $1.098631$ | |
| $h=0.125$ | $1.099768$ | $1.098620$ | $1.098613$ | $1.098613$ | $1.098613$ |

where the true solution is $1.098612$.

**Notes on Romberg**
The entries in the Romberg table are computed row-by-row, stopping when two successive diagonal-related entries in the table are sufficiently close together:

$$
\left|\frac{I_{k,k}-I_{k,k-1}}{I_{k,k}}\right| < \varepsilon.
$$

There is a convergence theorem: If 

$$\lim_{k\to\infty}I_{k,1} = \int_a^b f(x)dx,$$

then 

$$\lim_{k\to\infty}I_{k,k} = \int_a^b f(x)dx.$$

In [ ]:
import numpy as np
from math import erf, sqrt, pi

def print_row(row):
    # Print one row of the Romberg table in a fixed-width numeric format.
    # This makes the convergence pattern easy to read column by column.
    print(" ".join(f"{x:11.8f}" for x in row))
    
def Ex(x):
    return 1/x

def romberg(f, a, b, eps=1e-8, max_iter=20):
    """
    Approximate the definite integral of f from a to b using Romberg integration.

    Parameters
    ----------
    f : callable
        Function to integrate. It should ideally accept NumPy arrays so that
        evaluations at many points can be vectorized.
    a, b : float
        Lower and upper limits of integration.
    eps : float, optional
        Desired stopping tolerance. The algorithm stops when two successive
        diagonal Romberg estimates are close enough.
    max_iter : int, optional
        Maximum number of refinement levels allowed.

    Returns
    -------
    float
        The best Romberg estimate of the integral.
    """

    # I[n, m] will hold the nth refinement and the mth Richardson extrapolation.
    I = np.zeros((max_iter+1, max_iter+1))

    # Base case: trapezoidal rule using only the endpoints a and b.
    # This is the first and coarsest approximation to the integral.
    I[0, 0] = (b - a)*(f(a) + f(b))/2

    # Print the first row, which contains only the initial trapezoidal estimate.
    print_row(I[0, :1])

    # Refine the approximation level by level.
    # Each new level halves the step size and adds the function values at the
    # new midpoints introduced by subdivision.
    for k in range(1, max_iter+1):
        # Step size at refinement level k.
        # Since the interval is split into 2^k subintervals, the mesh width is:
        h_k1 = (b - a)/(2**k)   # h_{k-1}

        # Generate the new points introduced at this refinement level.
        # These are the odd multiples of h relative to a:
        # a + h, a + 3h, a + 5h, ..., a + (2^k - 1)h
        #
        # There are 2^(k-1) such points, and NumPy generates them efficiently.
        x = a + h_k1*np.arange(1, 2**k, 2)

        # Update the trapezoidal estimate using the recursive Romberg relation:.
        I[k, 0] =  I[k-1, 0]/2 + h_k1*np.sum(f(x))

        # Richardson extrapolation:
        for j in range(1, k+1):
            I[k, j] = I[k, j-1] + (I[k, j-1] - I[k-1, j-1])/(4**j - 1)

        # Print the completed row up to the current diagonal element.
        print_row(I[k, :k+1])

        # Stopping criterion:
        # If the last two diagonal-related values are sufficiently close,
        # we accept the most recent extrapolated estimate.
        if abs(I[k, k] - I[k, k-1]) < eps:
            return I[k, k]

    # If the method does not converge within max_iter levels, raise an error.
    raise RuntimeError("Romberg method did not converge within max_iter.")

# Compute and print the result.
print("Approximation =", romberg(Ex, 1.0, 3.0))
print("Exact value   =", np.log(3)-np.log(1))